# 📘 Phần 1: Xử Lý Dữ Liệu Trùng Lặp (Handling Duplicates in Pandas)

Dữ liệu trùng lặp (duplicates) là một trong những vấn đề dữ liệu phổ biến nhất. Trùng lặp có thể làm sai lệch các chỉ số thống kê (doanh thu, số lượng khách hàng, số lượng đơn hàng) và gây overfitting khi huấn luyện mô hình Machine Learning.

---

## 🎯 Mục Tiêu Bài Học:
1. Phát hiện trùng lặp hoàn toàn trên tất cả các cột (`df.duplicated()`).
2. Phát hiện trùng lặp theo khóa chính hoặc nhóm cột (`subset=['order_id']`).
3. Phân biệt và ứng dụng các tham số `keep='first'`, `keep='last'`, `keep=False`.
4. Phát hiện và xử lý **trùng lặp tiềm ẩn (Hidden Duplicates)** do viết hoa/thường, khoảng trắng.
5. Loại bỏ bản ghi trùng lặp an toàn với `drop_duplicates()`.


In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu mẫu từ thư mục data
df = pd.read_csv("data/customer_orders_raw.csv")
print(f"Tổng số dòng ban đầu: {len(df)}")
df.head(10)


## 1. Phát Hiện Dòng Trùng Lặp Hoàn Toàn (Exact Duplicates)
Sử dụng `df.duplicated()` để tìm các dòng có toàn bộ giá trị ở tất cả các cột giống hệt nhau.


In [ ]:
# Đếm tổng số dòng bị trùng lặp hoàn toàn
num_exact_dupes = df.duplicated().sum()
print(f"Số dòng trùng lặp 100%: {num_exact_dupes}")

# Hiển thị các dòng trùng lặp (giữ tất cả để đối chiếu)
df[df.duplicated(keep=False)]


## 2. Phát Hiện Trùng Lặp Theo Khóa Chính (Subset Duplicates)
Trong nhiều hệ thống, mã đơn hàng (`order_id`) hoặc số định danh khách hàng (`customer_id`, `email`) phải là duy nhất. Chúng ta dùng tham số `subset`.


In [ ]:
# Kiểm tra các dòng có order_id bị lặp lại
order_dupes = df[df.duplicated(subset=['order_id'], keep=False)]
print(f"Số bản ghi có order_id trùng lặp: {len(order_dupes)}")
order_dupes[['order_id', 'customer_name', 'email', 'price', 'payment_status']]


## 3. Các Tùy Chọn Của Tham Số `keep`
- `keep='first'` (mặc định): Giữ lại bản ghi đầu tiên, đánh dấu các bản ghi sau là trùng lặp.
- `keep='last'`: Giữ lại bản ghi xuất hiện cuối cùng (thường dùng khi muốn lấy trạng thái mới nhất).
- `keep=False`: Đánh dấu TẤT CẢ các bản ghi trùng lặp (dùng để cô lập toàn bộ các dòng nghi vấn để kiểm toán).


In [ ]:
# Giữ bản ghi đầu tiên
df_first = df.drop_duplicates(subset=['order_id'], keep='first')

# Giữ bản ghi cuối cùng
df_last = df.drop_duplicates(subset=['order_id'], keep='last')

print(f"Số dòng khi giữ first: {len(df_first)}")
print(f"Số dòng khi giữ last:  {len(df_last)}")


## 4. Xử Lý Trùng Lặp Tiềm Ẩn (Hidden Duplicates)
Hai khách hàng có thể cùng email hoặc tên nhưng máy tính không nhận diện được do:
- Ký tự viết hoa / viết thường (`TRAN.B@HOTMAIL.COM` vs `tran.b@hotmail.com`).
- Dấu cách thừa ở đầu hoặc cuối (`"  Le Van C  "`).

👉 **Giải pháp**: Tạo cột tạm thời đã được chuẩn hóa hoặc chuẩn hóa trực tiếp trước khi drop duplicates.


In [ ]:
# Chuẩn hóa email tạm thời để tìm trùng lặp tiềm ẩn
df['email_clean'] = df['email'].astype(str).str.strip().str.lower()

# Kiểm tra trùng lặp email
hidden_dupes = df[df.duplicated(subset=['email_clean'], keep=False)]
print(f"Số dòng trùng lặp email sau khi chuẩn hóa: {len(hidden_dupes)}")
hidden_dupes[['order_id', 'customer_name', 'email', 'email_clean']]


## 5. Thực Hiện Loại Bỏ Trùng Lặp Chuẩn Xác
Áp dụng loại bỏ trùng lặp và làm mới index của DataFrame.


In [ ]:
# Thực hiện loại bỏ trùng lặp theo order_id và giữ bản ghi đầu tiên
df_cleaned_dupes = df.drop_duplicates(subset=['order_id'], keep='first').copy()

# Xóa cột phụ tạm thời nếu có
df_cleaned_dupes.drop(columns=['email_clean'], inplace=True, errors='ignore')

# Reset lại chỉ số index
df_cleaned_dupes.reset_index(drop=True, inplace=True)

print(f"Kích thước ban đầu: {df.shape}")
print(f"Kích thước sau khi xử lý duplicates: {df_cleaned_dupes.shape}")
print("✅ Hoàn thành bài học xử lý trùng lặp!")
